In [2]:
pip install pandas

  Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
from datetime import date, timedelta
import requests
import zipfile

OUT_DIR = Path("ETHBTC_trades")
OUT_DIR.mkdir(exist_ok=True)

start = date(2024, 2, 1)
end   = date(2024, 4, 1)

current = start

while current <= end:
    ds = current.strftime("%Y-%m-%d")

    filename = f"ETHBTC-trades-{ds}.zip"
    url = (
        "https://data.binance.vision/data/spot/daily/trades/"
        f"ETHBTC/{filename}"
    )

    zip_path = OUT_DIR / filename

    print(f"Downloading {filename}")

    r = requests.get(url, stream=True, timeout=60)

    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        print(f"Extracting {filename}")

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(OUT_DIR)

        zip_path.unlink()  # delete zip after extraction

    else:
        print(f"Missing file: {filename}")

    current += timedelta(days=1)

print("Done.")

Extracting ETHBTC-trades-2024-02-01.zip
Extracting ETHBTC-trades-2024-02-02.zip
Extracting ETHBTC-trades-2024-02-03.zip
Extracting ETHBTC-trades-2024-02-04.zip
Extracting ETHBTC-trades-2024-02-05.zip
Extracting ETHBTC-trades-2024-02-06.zip
Extracting ETHBTC-trades-2024-02-07.zip
Extracting ETHBTC-trades-2024-02-08.zip
Extracting ETHBTC-trades-2024-02-09.zip
Extracting ETHBTC-trades-2024-02-10.zip
Extracting ETHBTC-trades-2024-02-11.zip
Extracting ETHBTC-trades-2024-02-12.zip
Extracting ETHBTC-trades-2024-02-13.zip
Extracting ETHBTC-trades-2024-02-14.zip
Extracting ETHBTC-trades-2024-02-15.zip
Extracting ETHBTC-trades-2024-02-16.zip
Extracting ETHBTC-trades-2024-02-17.zip
Extracting ETHBTC-trades-2024-02-18.zip
Extracting ETHBTC-trades-2024-02-19.zip
Extracting ETHBTC-trades-2024-02-20.zip
Extracting ETHBTC-trades-2024-02-21.zip
Extracting ETHBTC-trades-2024-02-22.zip
Extracting ETHBTC-trades-2024-02-23.zip
Extracting ETHBTC-trades-2024-02-24.zip
Extracting ETHBTC-trades-2024-02-25.zip


In [ ]:
from pathlib import Path
import pandas as pd

folder = Path("ETHBTC_trades")

dfs = []

for file in sorted(folder.glob("*.csv")):
    # Extract date from filename
    date_str = file.stem.split("-trades-")[1]

    df = pd.read_csv(
        file,
        header=None,
        names=[
            "trade_id",
            "price",
            "quantity",
            "quote_qty",
            "timestamp",
            "is_buyer_maker",
            "is_best_match"
        ]
    )

    df["date"] = pd.to_datetime(date_str)

    dfs.append(df)

trades = pd.concat(dfs, ignore_index=True)

print(trades.shape)
trades.head()

(7423472, 8)


,trade_id,price,quantity,quote_qty,timestamp,is_buyer_maker,is_best_match,date
0,431666550,0.05361,3.0000,0.160830,1706745603865,True,True,2024-02-01
1,431666551,0.05361,0.2185,0.011714,1706745604994,True,True,2024-02-01
2,431666552,0.05361,0.1341,0.007189,1706745604998,True,True,2024-02-01
3,431666553,0.05361,0.0204,0.001094,1706745605170,True,True,2024-02-01
4,431666554,0.05361,0.0874,0.004686,1706745606277,True,True,2024-02-01


: 

In [2]:
from pathlib import Path
from datetime import date, timedelta
import pandas as pd
import requests
import zipfile

# # ============================================================
# # DOWNLOAD
# # ============================================================

OUT_DIR = Path("ETHBTC_bookTicker")
# OUT_DIR.mkdir(exist_ok=True)

# start = date(2024, 2, 1)
# end = date(2024, 4, 1)

# current = start

# while current <= end:

#     ds = current.strftime("%Y-%m-%d")

#     filename = f"ETHBTC-bookTicker-{ds}.zip"

#     url = (
#         "https://data.binance.vision/"
#         "data/futures/um/daily/bookTicker/"
#         f"ETHBTC/{filename}"
#     )

#     zip_path = OUT_DIR / filename

#     print(f"Downloading {filename}")

#     r = requests.get(url, stream=True)

#     if r.status_code == 200:

#         with open(zip_path, "wb") as f:
#             for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
#                 if chunk:
#                     f.write(chunk)

#         with zipfile.ZipFile(zip_path, "r") as z:
#             z.extractall(OUT_DIR)

#         zip_path.unlink()

#     else:
#         print(f"Missing: {filename}")

#     current += timedelta(days=1)

# print("Download complete.")

# ============================================================
# LOAD + MERGE
# ============================================================

dfs = []

for file in sorted(OUT_DIR.glob("*.csv")):

    date_str = file.stem.split("-bookTicker-")[1]

    df = pd.read_csv(file)

    df["file_date"] = pd.to_datetime(date_str)

    dfs.append(df)

book = pd.concat(dfs, ignore_index=True)

# ============================================================
# TIMESTAMPS
# ============================================================

book["transaction_dt"] = pd.to_datetime(
    book["transaction_time"],
    unit="ms",
    utc=True
)

book["event_dt"] = pd.to_datetime(
    book["event_time"],
    unit="ms",
    utc=True
)

# ============================================================
# OPTIMIZE MEMORY
# ============================================================

float_cols = [
    "best_bid_price",
    "best_bid_qty",
    "best_ask_price",
    "best_ask_qty"
]

for c in float_cols:
    book[c] = book[c].astype("float32")

book["update_id"] = book["update_id"].astype("int64")

# ============================================================
# DERIVED FEATURES
# ============================================================

book["mid"] = (
    book["best_bid_price"] +
    book["best_ask_price"]
) / 2

book["spread"] = (
    book["best_ask_price"] -
    book["best_bid_price"]
)

book["imbalance"] = (
    book["best_bid_qty"] -
    book["best_ask_qty"]
) / (
    book["best_bid_qty"] +
    book["best_ask_qty"] + 1e-9
)

print(book.shape)
book.head()

: 

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 1. SPOT TRADES
# ============================================================

# timestamp is in MICROSECONDS
spot["dt"] = pd.to_datetime(
    spot["timestamp"],
    unit="us",
    utc=True
)

# Buyer initiated volume
spot["buy_qty"] = np.where(
    ~spot["is_buyer_maker"],
    spot["quantity"],
    0
)

# Seller initiated volume
spot["sell_qty"] = np.where(
    spot["is_buyer_maker"],
    spot["quantity"],
    0
)

spot_100ms = (
    spot
    .set_index("dt")
    .resample("100ms")
    .agg(
        open=("price", "first"),
        high=("price", "max"),
        low=("price", "min"),
        close=("price", "last"),
        volume=("quantity", "sum"),
        quote_volume=("quote_qty", "sum"),
        qty_b=("buy_qty", "sum"),
        qty_s=("sell_qty", "sum"),
        trade_count=("trade_id", "count")
    )
)

# ============================================================
# 2. FUTURES BOOKTICKER
# ============================================================

# event_time is in MILLISECONDS
futures["dt"] = pd.to_datetime(
    futures["event_time"],
    unit="ms",
    utc=True
)

# Midprice
futures["mid"] = (
    futures["best_bid_price"]
    + futures["best_ask_price"]
) / 2

# Spread
futures["spread"] = (
    futures["best_ask_price"]
    - futures["best_bid_price"]
)

# Queue imbalance
futures["imbalance"] = (
    futures["best_bid_qty"]
    - futures["best_ask_qty"]
) / (
    futures["best_bid_qty"]
    + futures["best_ask_qty"]
    + 1e-9
)

futures_100ms = (
    futures
    .set_index("dt")
    .resample("100ms")
    .agg(
        bid=("best_bid_price", "last"),
        ask=("best_ask_price", "last"),
        bid_qty=("best_bid_qty", "last"),
        ask_qty=("best_ask_qty", "last"),
        mid=("mid", "last"),
        spread=("spread", "last"),
        imbalance=("imbalance", "last")
    )
)

# ============================================================
# 3. MERGE
# ============================================================

df = pd.merge(
    spot_100ms,
    futures_100ms,
    left_index=True,
    right_index=True,
    how="outer"
)

# ============================================================
# 4. FILL BOOK STATE
# ============================================================

book_cols = [
    "bid",
    "ask",
    "bid_qty",
    "ask_qty",
    "mid",
    "spread",
    "imbalance"
]

df[book_cols] = df[book_cols].ffill()

# ============================================================
# 5. FILL TRADE DATA
# ============================================================

trade_cols = [
    "volume",
    "quote_volume",
    "qty_b",
    "qty_s",
    "trade_count"
]

df[trade_cols] = df[trade_cols].fillna(0)

# For OHLC, use previous close if no trade occurred
df["close"] = df["close"].ffill()

df["open"] = df["open"].fillna(df["close"])
df["high"] = df["high"].fillna(df["close"])
df["low"] = df["low"].fillna(df["close"])

# ============================================================
# 6. FEATURE ENGINEERING
# ============================================================

# Trade imbalance
df["trade_imbalance"] = (
    df["qty_b"] - df["qty_s"]
) / (
    df["qty_b"] + df["qty_s"] + 1e-9
)

# VWAP
df["vwap"] = np.where(
    df["volume"] > 0,
    df["quote_volume"] / df["volume"],
    np.nan
)

df["vwap"] = df["vwap"].ffill()

# Futures premium / discount
df["basis"] = (
    df["mid"] - df["close"]
) / df["close"]

# Spot return
df["ret_100ms"] = df["close"].pct_change()

# Mid return
df["mid_ret_100ms"] = df["mid"].pct_change()

# Realized range
df["range"] = (
    df["high"] - df["low"]
) / df["close"]

# ============================================================
# 7. TARGETS
# ============================================================

# Predict next 100ms spot return
df["target_ret_100ms"] = (
    df["close"].shift(-1) - df["close"]
) / df["close"]

# Predict next 100ms futures mid return
df["target_mid_ret_100ms"] = (
    df["mid"].shift(-1) - df["mid"]
) / df["mid"]

# Direction classification target
df["target_direction"] = np.sign(
    df["target_ret_100ms"]
)

# ============================================================
# 8. CLEAN
# ============================================================

df = df.dropna()

print(df.shape)
print(df.head())

# df is now your final ML dataset